In [1]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql import types

In [2]:
spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/01 11:31:17 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


**Q1**: Install Spark and PySpark

In [3]:
spark.version

'4.1.1'

**Q2**: What is the average size of the Parquet (ending with .parquet extension) Files that were created (in MB)?

In [4]:
import os

file_path = 'yellow_tripdata_2025-11.parquet'
file_size_bytes = os.path.getsize(file_path)
#file_size_mb = file_size_bytes / (1024 * 1024)
print(f"File size: {file_size_bytes:.2f} MB")

File size: 71134255.00 MB


In [5]:
df = spark.read.parquet(file_path)

In [6]:
df.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)



**Q3**: How many taxi trips were there on November 15?

In [7]:
from pyspark.sql import functions as F

In [8]:
df \
    .withColumn('pickup_date', F.to_date(df.tpep_pickup_datetime)) \
    .filter("pickup_date = '2025-11-15'") \
    .count()

162604

In [9]:
df.registerTempTable('fhvhv_2025_11')

/Users/lilianateixeira/Desktop/Portefolio/data_engeneering_course/data-engineering-course/module-1/pipeline/.venv/lib/python3.13/site-packages/pyspark/sql/classic/dataframe.py:178: FutureWarning: Deprecated in 2.0, use createOrReplaceTempView instead.
  warnings.warn("Deprecated in 2.0, use createOrReplaceTempView instead.", FutureWarning)


In [10]:
df.show()

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|       7| 2025-11-01 00:13:25|  2025-11-01 00:13:25|              1|         1.68|         1|                 N|          43|    

**Q4**: What is the length of the longest trip in the dataset in hours?

In [19]:
spark.sql("""
SELECT
  round(max((unix_timestamp(tpep_dropoff_datetime) - unix_timestamp(tpep_pickup_datetime)) / 3600.0), 1) AS trip_duration_hours
FROM
  fhvhv_2025_11
""").show()

+-------------------+
|trip_duration_hours|
+-------------------+
|               90.6|
+-------------------+



**Q4**: Spark's User Interface which shows the application's dashboard runs on which local port?

In [20]:
import re

print(spark.sparkContext.uiWebUrl)

url = spark.sparkContext.uiWebUrl
port = re.search(r":(\d+)", url).group(1) if url else None
print("UI port:", port)

http://macbookair.home:4040
UI port: 4040


**Q6**: Using the zone lookup data and the Yellow November 2025 data, what is the name of the LEAST frequent pickup location Zone?

In [22]:
!ls -lh taxi_zone_lookup.csv

-rw-r--r--@ 1 lilianateixeira  staff    12K Feb 22  2024 taxi_zone_lookup.csv


In [27]:
taxi_data = spark.read \
    .option("header", "true") \
    .csv('taxi_zone_lookup.csv')

taxi_data.show()
taxi_data.registerTempTable('taxi_data_zones')

+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
|         6|Staten Island|Arrochar/Fort Wad...|   Boro Zone|
|         7|       Queens|             Astoria|   Boro Zone|
|         8|       Queens|        Astoria Park|   Boro Zone|
|         9|       Queens|          Auburndale|   Boro Zone|
|        10|       Queens|        Baisley Park|   Boro Zone|
|        11|     Brooklyn|          Bath Beach|   Boro Zone|
|        12|    Manhattan|        Battery Park| Yellow Zone|
|        13|    Manhattan|   Battery Park City| Yellow Zone|
|        14|     Brookly

/Users/lilianateixeira/Desktop/Portefolio/data_engeneering_course/data-engineering-course/module-1/pipeline/.venv/lib/python3.13/site-packages/pyspark/sql/classic/dataframe.py:178: FutureWarning: Deprecated in 2.0, use createOrReplaceTempView instead.
  warnings.warn("Deprecated in 2.0, use createOrReplaceTempView instead.", FutureWarning)


In [26]:
df.show()

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|       7| 2025-11-01 00:13:25|  2025-11-01 00:13:25|              1|         1.68|         1|                 N|          43|    

In [ ]:
spark.sql("""
SELECT Zone, count(*) as trip_count FROM (
    SELECT t.*,
        z.Borough,
        z.Zone
    FROM fhvhv_2025_11 t
    JOIN taxi_data_zones z
    ON t.PULocationID = CAST(z.LocationID AS INT))
GROUP BY Zone
ORDER BY count(*) ASC
LIMIT 5
""").show()

+--------------------+
|                Zone|
+--------------------+
|Governor's Island...|
|Eltingville/Annad...|
|       Arden Heights|
|       Port Richmond|
|       Rikers Island|
+--------------------+

